# Imports

In [ ]:
import json
import shutil
import sys

import numpy as np
import onnxruntime as ort
import torch

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.imagenet import normalise
from uavloc.features.segmentation import load_segmenter
from uavloc.track import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
CHECKPOINT_DIR = REPO_ROOT / "pocs" / "mobilenet-lraspp-speed"
OUT_DIR = REPO_ROOT / "pocs" / "lraspp-pi5" / "models"
DERIVED_DIR = REPO_ROOT / "data" / "derived"
FRAME_DIR = OUT_DIR / "frames"
CHECKPOINTS = {"drone": "model-lraspp-bn-drone.pt", "satellite": "model-lraspp-bn-satellite.pt"}
INPUT_SHAPES = {"drone": [1, 3, 416, 736], "satellite": [1, 3, 512, 512]}
OPSET = 17
SEED = 0
DRONE_VAL_BLOCKS = [0, 4]
DRONE_VAL_INDEX = 0
DRONE_GSD_M = 0.10
HOLDOUT_VILLAGE = "bushcha-50p2422-25p8305"
SAT_DATE = "current"

# Data

In [ ]:
drone = vs.load_drone(DERIVED_DIR / "ua-drone-annotated", DERIVED_DIR)
vs.split_by_footprint(drone, DRONE_VAL_BLOCKS)
drone_rgb, drone_label = vs.drone_at_gsd([d for d in drone if d.split == "val"][DRONE_VAL_INDEX], DRONE_GSD_M)
drone_rgb.shape

In [ ]:
sat = next(s for s in vs.load_satellite(DERIVED_DIR / "ua-village-annotated", DERIVED_DIR)
           if s.loc == HOLDOUT_VILLAGE and s.date == SAT_DATE)
sat.img.shape

# Model

In [ ]:
def export_onnx(model: torch.nn.Module, x: torch.Tensor, path) -> None:
    torch.onnx.export(model, x, str(path), input_names=["image"], output_names=["logits"], opset_version=OPSET)

In [ ]:
def onnx_max_abs_diff(path, x: np.ndarray, reference: np.ndarray) -> float:
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    return float(np.abs(session.run(None, {"image": x})[0] - reference).max())

In [ ]:
@torch.no_grad()
def export_one(name: str) -> dict:
    model = load_segmenter(CHECKPOINT_DIR / CHECKPOINTS[name], "cpu", architecture="lraspp")
    x = torch.randn(*INPUT_SHAPES[name], generator=torch.Generator().manual_seed(SEED))
    reference = model(x).numpy()
    onnx_path = OUT_DIR / f"lraspp-{name}.onnx"
    export_onnx(model, x, onnx_path)
    np.savez(OUT_DIR / f"reference-{name}.npz", x=x.numpy(), logits=reference)
    shutil.copy(CHECKPOINT_DIR / CHECKPOINTS[name], OUT_DIR / f"lraspp-{name}.pt")
    return {"model": name, "checkpoint": CHECKPOINTS[name], "input_shape": INPUT_SHAPES[name],
            "onnx_max_abs_diff": onnx_max_abs_diff(onnx_path, x.numpy(), reference)}

In [ ]:
def fit_to_input(rgb: np.ndarray, label: np.ndarray, name: str) -> tuple[np.ndarray, np.ndarray]:
    """Centre-crop to the model's fixed input, then reflect-pad (label padded with IGNORE_LABEL) where the image is smaller."""
    h, w = INPUT_SHAPES[name][2:]
    top, left = max(0, (rgb.shape[0] - h) // 2), max(0, (rgb.shape[1] - w) // 2)
    rgb, label = rgb[top:top + h, left:left + w], label[top:top + h, left:left + w]
    pad = ((0, h - rgb.shape[0]), (0, w - rgb.shape[1]))
    return np.pad(rgb, (*pad, (0, 0)), mode="reflect"), np.pad(label, pad, constant_values=IGNORE_LABEL)

In [ ]:
def save_sample(name: str, rgb: np.ndarray, label: np.ndarray) -> list[int]:
    """Real input for the Pi mask check: `x` is `(1, 3, H, W)` ImageNet-normalised float32."""
    rgb, label = fit_to_input(rgb, label, name)
    x = normalise(rgb).transpose(2, 0, 1)[None].copy()
    np.savez(OUT_DIR / f"sample-{name}.npz", x=x, rgb=rgb, label=label)
    return list(x.shape)

In [ ]:
def export_frames(names: list[str]) -> int:
    """Raw drone frames + `frames.jsonl` (file, mpp) for the Pi pipeline notebook."""
    records = {r["location"]: r for r in vs.read_jsonl(DERIVED_DIR / "ua-drone-annotated" / "samples.jsonl")}
    FRAME_DIR.mkdir(parents=True, exist_ok=True)
    rows = []
    for name in names:
        source = DERIVED_DIR / records[name]["image_file"]
        shutil.copy(source, FRAME_DIR / f"{name}{source.suffix}")
        rows.append({"name": name, "file": f"{name}{source.suffix}", "mpp": records[name]["mpp"]})
    (FRAME_DIR / "frames.jsonl").write_text("".join(json.dumps(row) + "\n" for row in rows))
    return len(rows)

# Run

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
exports = [export_one(name) for name in CHECKPOINTS]
exports

In [ ]:
(OUT_DIR / "export.json").write_text(json.dumps(
    {"opset": OPSET, "torch": torch.__version__, "onnxruntime": ort.__version__, "exports": exports}, indent=2))

In [ ]:
{"drone": save_sample("drone", drone_rgb, drone_label), "satellite": save_sample("satellite", sat.img, sat.lab)}

In [ ]:
export_frames([d.name for d in drone if d.split == "val"])